# Masked Pokemon Team Transformer

Predicts a masked team member (species, ability, item, moveset) given the other 5 Pokemon on a team.

Each Pokemon is encoded as:

`Species Learned Emb | Species RDV | Ability Learned Emb | Item Learned Emb | Item RDV | Moveset Learned Emb | Moveset RDV`

(Ability has only a learned embedding; no ability RDV was prepared.) The moveset embedding/RDV are the
mean across the Pokemon's 4 moves, with an optional multi-head attention layer applied over the 4 moves
before averaging.

In [1]:
import pickle
import random
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import os

## Colab Cells

Mount Google Drive and define the shared-drive directory prefixes. When running locally, comment out
the drive mount and point these prefixes at local paths instead.

In [2]:
from google.colab import drive
from google.colab import runtime
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ---- Shared-drive directory roots ----
TEAM_DIR    = '/content/drive/Shared drives/ML2 Final Project/Data/Scraped Pokemon Teams/'
VECTOR_DIR  = '/content/drive/Shared drives/ML2 Final Project/Data/Transformer Ready Vectors/'
DICT_DIR    = '/content/drive/Shared drives/ML2 Final Project/Data/Game Info Dictionaries/'
MODEL_DIR   = '/content/drive/Shared drives/ML2 Final Project/Models'
RESULTS_DIR = '/content/drive/Shared drives/ML2 Final Project/Gridsearch Results'

## Configuration

Edit this cell to change embedding sizes, transformer size, masking, and training settings.

In [22]:
# ---- Learned-embedding size presets (species / ability / item / move) ----
EMBED_CONFIGS = {
    "small":  {"species": 16, "ability": 8,  "item": 8,  "move": 16},
    "medium": {"species": 32, "ability": 16, "item": 16, "move": 32},
    "large":  {"species": 64, "ability": 32, "item": 32, "move": 64},
}
EMBED_CONFIG_NAME = "medium"          # one of: small | medium | large
EMB = EMBED_CONFIGS[EMBED_CONFIG_NAME]

# ---- Raw-data-vector dimensionalities (fixed by the prepared .pkl files) ----
SPECIES_RDV_DIM = 42
ITEM_RDV_DIM    = 6
MOVE_RDV_DIM    = 44

# ---- Moveset aggregation ----
USE_MOVE_ATTENTION = True           # if True, run MHA over the 4 moves before averaging
MOVE_ATTENTION_HEADS = 2

# ---- Transformer ----
D_MODEL          = 128
N_HEADS          = 8
N_LAYERS         = 4
DIM_FEEDFORWARD  = 256
DROPOUT          = 0.1

# ---- Masking ----
TRAIN_MASK_STRATEGY = "random"       # "random" = mask any of the 6 (augmentation); "last" = always 6th
EVAL_MASK_STRATEGY  = "last"         # held-out evaluation always predicts the 6th slot

# ---- Training ----
BATCH_SIZE   = 32
EPOCHS       = 50
LR           = 1e-3
SEED         = 42
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

# ---- Debug-run override (Section 8) ----
DEBUG_RUN          = False          # if True, use a tiny 50-team sample to smoke-test the pipeline
DEBUG_TOTAL_TEAMS  = 50
DEBUG_TEST_TEAMS   = 10

# ---- Multi-Pokemon recommendation: how many candidate predictions per slot ----
# 1 = original behavior (top-1 + standard CE). n>1 = best-of-n training loss
# (zero loss when truth is among the n predictions; standard CE otherwise) and
# any-of-n accuracy at eval/inspection time.
TRAIN_PREDICTIONS = 1
TEST_PREDICTIONS  = 6

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
set_seed(SEED)
print("Config:", EMBED_CONFIG_NAME, EMB, "| device:", DEVICE)

Config: medium {'species': 32, 'ability': 16, 'item': 16, 'move': 32} | device: cuda


In [21]:
# ---- Legality enforcement & sampling (Section 6 features) ----
# The model can verify a predicted Pokemon actually has the predicted ability/moves
# (using pokemon_dict) and self-correct to the best *legal* option.
ENFORCE_LEGALITY = True      # mask ability/move predictions to the predicted species' legal set

# Decoding for the SPECIES head:
#   TOP_K = 0            -> top-k disabled
#   TOP_K >= 1           -> keep K highest-logit species, softmax, sample
#   TOP_P = 0  (or >=1)  -> top-p (nucleus) disabled
#   0 < TOP_P < 1        -> keep smallest species set with cumulative prob >= TOP_P, softmax, sample
# If both are set, top-k is applied first, then top-p within it. When sampling is
# inactive the species is taken as argmax. Ability/moveset are then predicted from a
# distribution restricted to ONLY what the chosen species can legally have.
TOP_K = 0
TOP_P = 0
TEMP = 1.0

# When the top-k/p sampling pipeline is active:
#   "off"     -> never (argmax everywhere; legality still applies if ENFORCE_LEGALITY)
#   "train"   -> only while updating weights
#   "predict" -> only at inference / evaluation
#   "both"    -> training and inference
SAMPLING_PHASE = "both"

## 1. Data Loading

Raw-data-vector dictionaries (`pokemon_vectors.pkl`, `item_vectors.pkl`, `move_vectors.pkl`) come from
`prepare_raw_vectors.ipynb`. Team pools are the scraped `*_team_vectors*.pkl` files. Change `POOL_FILES`
to alter which sources feed the model.

In [6]:
with open(VECTOR_DIR + "pokemon_vectors.pkl", "rb") as f: SPECIES_RDV = pickle.load(f)
with open(VECTOR_DIR + "item_vectors.pkl",    "rb") as f: ITEM_RDV    = pickle.load(f)
with open(VECTOR_DIR + "move_vectors.pkl",    "rb") as f: MOVE_RDV    = pickle.load(f)
with open(DICT_DIR   + "ability_dict.pkl",    "rb") as f: ABILITY_DICT = pickle.load(f)
with open(DICT_DIR   + "pokemon_dict.pkl",    "rb") as f: POKEMON_DICT = pickle.load(f)

# Each pool file -> list of teams; each team -> 6 Pokemon;
# each Pokemon -> [species, ability, item, move1, move2, move3, move4]
POOL_FILES = {
    "vgcpastes":  "team_vectors.pkl",
    "vgenc":      "vgenc_team_vectors.pkl",
    "limitless":  "limitless_team_vectors.pkl",
}
TEAM_POOLS = {}
for name, path in POOL_FILES.items():
    with open(TEAM_DIR + path, "rb") as f:
        TEAM_POOLS[name] = pickle.load(f)
    print(f"{name:11s}: {len(TEAM_POOLS[name])} teams")

vgcpastes  : 769 teams
vgenc      : 2611 teams
limitless  : 6109 teams


In [7]:
for k in TEAM_POOLS.keys():
    for t in TEAM_POOLS[k]:
        if len(t) != 6:
            print(k)
            print(t)
            print(len(t))

## 2. Mega Stone Handling

If a Pokemon holds a Mega Stone, its species is renamed to the corresponding Mega form so it receives
the Mega's base stats / typing RDV. The held item is left unchanged. Charizard (X/Y) is handled
explicitly.

In [8]:
MEGA_STONE_TO_FORM = {
    "Abomasite": "Abomasnow-Mega", "Absolite": "Absol-Mega",
    "Aerodactylite": "Aerodactyl-Mega", "Aggronite": "Aggron-Mega",
    "Alakazite": "Alakazam-Mega", "Altarianite": "Altaria-Mega",
    "Ampharosite": "Ampharos-Mega", "Audinite": "Audino-Mega",
    "Banettite": "Banette-Mega", "Beedrillite": "Beedrill-Mega",
    "Blastoisinite": "Blastoise-Mega", "Cameruptite": "Camerupt-Mega",
    "Chandelurite": "Chandelure-Mega",
    "Charizardite X": "Charizard-Mega-X", "Charizardite Y": "Charizard-Mega-Y",
    "Chesnaughtite": "Chesnaught-Mega", "Chimechite": "Chimecho-Mega",
    "Clefablite": "Clefable-Mega", "Crabominite": "Crabominable-Mega",
    "Delphoxite": "Delphox-Mega", "Dragoninite": "Dragonite-Mega",
    "Drampanite": "Drampa-Mega", "Emboarite": "Emboar-Mega",
    "Excadrite": "Excadrill-Mega", "Feraligite": "Feraligatr-Mega",
    "Floettite": "Floette-Mega", "Froslassite": "Froslass-Mega",
    "Galladite": "Gallade-Mega", "Garchompite": "Garchomp-Mega",
    "Gardevoirite": "Gardevoir-Mega", "Gengarite": "Gengar-Mega",
    "Glalitite": "Glalie-Mega", "Glimmoranite": "Glimmora-Mega",
    "Golurkite": "Golurk-Mega", "Greninjite": "Greninja-Mega",
    "Gyaradosite": "Gyarados-Mega", "Hawluchanite": "Hawlucha-Mega",
    "Heracronite": "Heracross-Mega", "Houndoominite": "Houndoom-Mega",
    "Kangaskhanite": "Kangaskhan-Mega", "Lopunnite": "Lopunny-Mega",
    "Lucarionite": "Lucario-Mega", "Manectite": "Manectric-Mega",
    "Medichamite": "Medicham-Mega", "Meganiumite": "Meganium-Mega",
    "Meowsticite": "Meowstic-Mega", "Pidgeotite": "Pidgeot-Mega",
    "Pinsirite": "Pinsir-Mega", "Sablenite": "Sableye-Mega",
    "Scizorite": "Scizor-Mega", "Scovillainite": "Scovillain-Mega",
    "Sharpedonite": "Sharpedo-Mega", "Skarmorite": "Skarmory-Mega",
    "Slowbronite": "Slowbro-Mega", "Starminite": "Starmie-Mega",
    "Steelixite": "Steelix-Mega", "Tyranitarite": "Tyranitar-Mega",
    "Venusaurite": "Venusaur-Mega", "Victreebelite": "Victreebel-Mega",
}

import re
_PAREN_RE = re.compile(r"\(([^()]+)\)")

def resolve_nickname(species):
    """If `species` is not in POKEMON_DICT but contains a parenthesized
    sub-string that IS a real Pokemon name, return that name. Handles
    paste entries like "Recto/Verso (Gardevoir)" or "Warding Chime (Chimecho-Mega) (M)".
    """
    if species in POKEMON_DICT:
        return species
    for match in _PAREN_RE.findall(species):
        cand = match.strip()
        if cand in POKEMON_DICT:
            return cand
    return species

def apply_mega(pokemon):
    """Return a copy of one Pokemon list with species renamed to its Mega form
    when it holds a Mega Stone (and that Mega form exists in the species RDVs)."""
    species, ability, item = pokemon[0], pokemon[1], pokemon[2]
    mega = MEGA_STONE_TO_FORM.get(item)
    if mega is not None and mega in SPECIES_RDV:
        p = list(pokemon)
        p[0] = mega
        return p
    return list(pokemon)
def apply_variant(pokemon):
    """Return a copy of one Pokemon list with species renamed to its Mega form
    when it holds a Mega Stone (and that Mega form exists in the species RDVs)."""
    species, ability, item = pokemon[0], pokemon[1], pokemon[2]
    mega = MEGA_STONE_TO_FORM.get(item)
    if mega is not None and mega in SPECIES_RDV:
        p = list(pokemon)
        p[0] = mega
        return p
    return list(pokemon)
def normalize_team(team):
    out = []
    for p in team:
        p = list(p)
        p[0] = resolve_nickname(p[0])      # un-nickname first
        out.append(apply_mega(p))          # then promote to Mega form if holding a stone
    return out

## 3. Vocabularies

Learned-embedding vocabularies are built from every name observed across **all** pools (so train and
held-out teams share the index space) unioned with the RDV dictionary keys. Index 0 is reserved for
`<UNK>` (covers names with no RDV / unseen entries). RDV lookups fall back to a zero vector when a
name is absent, and missing fractions are reported.

In [9]:
UNK = "<UNK>"

def build_vocab(values):
    vocab = {UNK: 0}
    for v in values:
        if v not in vocab:
            vocab[v] = len(vocab)
    return vocab

_sp, _ab, _it, _mv = set(SPECIES_RDV), set(ABILITY_DICT), set(ITEM_RDV), set(MOVE_RDV)
for pool in TEAM_POOLS.values():
    for team in pool:
        for p in normalize_team(team):
            _sp.add(p[0]); _ab.add(p[1]); _it.add(p[2])
            for m in p[3:7]: _mv.add(m)

SPECIES_VOCAB = build_vocab(sorted(_sp))
ABILITY_VOCAB = build_vocab(sorted(_ab))
ITEM_VOCAB    = build_vocab(sorted(_it))
MOVE_VOCAB    = build_vocab(sorted(_mv))
N_SPECIES, N_ABILITY = len(SPECIES_VOCAB), len(ABILITY_VOCAB)
N_ITEM, N_MOVE       = len(ITEM_VOCAB), len(MOVE_VOCAB)
print(f"vocab sizes -> species {N_SPECIES}, ability {N_ABILITY}, item {N_ITEM}, move {N_MOVE}")

ZERO_SPECIES_RDV = [0.0] * SPECIES_RDV_DIM
ZERO_ITEM_RDV    = [0.0] * ITEM_RDV_DIM
ZERO_MOVE_RDV    = [0.0] * MOVE_RDV_DIM

# RDV coverage diagnostic over normalized teams
tot = miss_s = miss_i = miss_m = 0
for pool in TEAM_POOLS.values():
    for team in pool:
        for p in normalize_team(team):
            tot += 1
            if p[0] not in SPECIES_RDV: miss_s += 1
            if p[2] not in ITEM_RDV:    miss_i += 1
            for m in p[3:7]:
                if m not in MOVE_RDV:   miss_m += 1
print(f"RDV miss rate -> species {miss_s/tot:.1%}, item {miss_i/tot:.1%}, move {miss_m/(tot*4):.1%}")

vocab sizes -> species 336, ability 269, item 209, move 683
RDV miss rate -> species 7.4%, item 0.9%, move 1.2%


## 4. Team -> Tensor Conversion

`team_to_tensors` maps a raw nested-list team (post Mega-rename) to index tensors and RDV tensors by
looking each name up in its vocabulary / RDV dictionary. `MaskedTeamDataset` then picks the masked
slot per sample and exposes the prediction targets.

In [10]:
def team_to_tensors(team):
    """team: list of 6 Pokemon lists (already Mega-normalized).
    Returns a dict of tensors describing all 6 Pokemon."""
    sp_idx, ab_idx, it_idx = [], [], []
    mv_idx, sp_rdv, it_rdv, mv_rdv = [], [], [], []
    for p in team:
        species, ability, item = p[0], p[1], p[2]
        moves = list(p[3:7]) + [""] * (4 - len(p[3:7]))
        sp_idx.append(SPECIES_VOCAB.get(species, 0))
        ab_idx.append(ABILITY_VOCAB.get(ability, 0))
        it_idx.append(ITEM_VOCAB.get(item, 0))
        mv_idx.append([MOVE_VOCAB.get(m, 0) for m in moves])
        sp_rdv.append(SPECIES_RDV.get(species, ZERO_SPECIES_RDV))
        it_rdv.append(ITEM_RDV.get(item, ZERO_ITEM_RDV))
        mv_rdv.append([MOVE_RDV.get(m, ZERO_MOVE_RDV) for m in moves])
    return {
        "species_idx": torch.tensor(sp_idx, dtype=torch.long),       # [6]
        "ability_idx": torch.tensor(ab_idx, dtype=torch.long),       # [6]
        "item_idx":    torch.tensor(it_idx, dtype=torch.long),       # [6]
        "move_idx":    torch.tensor(mv_idx, dtype=torch.long),       # [6,4]
        "species_rdv": torch.tensor(sp_rdv, dtype=torch.float),      # [6,42]
        "item_rdv":    torch.tensor(it_rdv, dtype=torch.float),      # [6,6]
        "move_rdv":    torch.tensor(mv_rdv, dtype=torch.float),      # [6,4,44]
    }

class MaskedTeamDataset(Dataset):
    def __init__(self, teams, mask_strategy):
        # store normalized teams
        self.teams = [normalize_team(t) for t in teams]
        self.mask_strategy = mask_strategy

    def __len__(self):
        return len(self.teams)

    def __getitem__(self, i):
        team = self.teams[i]
        t = team_to_tensors(team)
        if self.mask_strategy == "random":
            mpos = random.randrange(6)
        else:                       # "last"
            mpos = 5
        t["mask_pos"] = torch.tensor(mpos, dtype=torch.long)
        # targets = the masked Pokemon's attributes
        t["y_species"] = t["species_idx"][mpos].clone()
        t["y_ability"] = t["ability_idx"][mpos].clone()
        t["y_item"]    = t["item_idx"][mpos].clone()
        mv_multi = torch.zeros(N_MOVE)
        for mi in t["move_idx"][mpos].tolist():
            mv_multi[mi] = 1.0
        t["y_moves"] = mv_multi                                    # multi-hot moveset
        return t

## 5. Train / Test Split

Edit this section to change the pool or the split. By default the **test set is a random 20% of the
`limitless` teams**; everything else (`vgcpastes`, `vgenc`, and the remaining 80% of `limitless`)
is training. The debug override (Section 8) instead carves a 50-team sample.

In [11]:
TEST_SOURCE   = "limitless"
TEST_FRACTION = 0.20

def make_split(team_pools, test_source=TEST_SOURCE, test_fraction=TEST_FRACTION, seed=SEED):
    rng = random.Random(seed)
    src = list(team_pools[test_source])
    idx = list(range(len(src)))
    rng.shuffle(idx)
    n_test = int(round(len(src) * test_fraction))
    test_ids = set(idx[:n_test])
    test_teams  = [src[i] for i in idx[:n_test]]
    train_teams = [src[i] for i in idx[n_test:]]
    for name, pool in team_pools.items():
        if name == test_source:
            continue
        train_teams.extend(pool)
    rng.shuffle(train_teams)
    return train_teams, test_teams

def make_debug_split(team_pools, total=DEBUG_TOTAL_TEAMS, n_test=DEBUG_TEST_TEAMS,
                     source=TEST_SOURCE, seed=SEED):
    rng = random.Random(seed)
    src = list(team_pools[source])
    rng.shuffle(src)
    sample = src[:total]
    return sample[n_test:], sample[:n_test]

if DEBUG_RUN:
    train_teams, test_teams = make_debug_split(TEAM_POOLS)
else:
    train_teams, test_teams = make_split(TEAM_POOLS)
print(f"train teams: {len(train_teams)} | test teams: {len(test_teams)}")

train teams: 8267 | test teams: 1222


In [12]:
full_teams = train_teams + test_teams

In [13]:
team_mon_list = [[p[0] for p in t] for t in full_teams]
names_in_teams = set([item for sublist in team_mon_list for item in sublist])

In [14]:
print("Appears in team DB but not in Pokemon Dict Keys")
for n in names_in_teams:
    if n not in POKEMON_DICT.keys():
        print(n)
print("Appears in Pokemon Dict Keys but not in team DB")
for k in POKEMON_DICT.keys():
    if k not in names_in_teams:
        print(k)

Appears in team DB but not in Pokemon Dict Keys
Goodra-Hisui
Hisuian Decidueye
Salazzle (F)
Farigiraf (F)
Incineroar (M)
Decidueye-Hisui
I CONNECT (Rotom-Heat)
Meowstic ♀
Gourgeist-Small
Galarian Slowbro
Mega Scovillain
Rotom-Frost
Zoroark-Hisui
Mega Greninja
Torkoal (M)
Mega Lopunny
Heat Rotom
Mega Skarmory
Tinkaton (F)
Empress (Lopunny-Mega)
Galarian Stunfisk
Meowscarada (F)
Mega Dragonite
Basculegion ♀
Emperor (Kingambit)
Hisuian Typhlosion
Mega Charizard Y
Aegislash Blade Forme
Courier (Pelipper)
Howler (Talonflame)
Basculegion-F
Swanging (Incineroar) (M)
Lycanroc-Dusk
Charizard-Mega-Y-Mega-Y
Méga-Soléil (Meganium-Mega) (F)
Mega Gyarados
Lycanroc Dusk
Mega Blastoise
Froslass (F)
Alcremie (F)
Hisuian Arcanine
Chai tea (Sinistcha)
Charizard-Mega-Y (M)
Talonflame (M)
Hisuian Goodra
Paldean Tauros Aqua Breed
Scovillain (F)
Arcanine-Hisui
Floette-Eternal-Mega
Tauros-Paldea-Blaze (M)
Paldean Tauros
Samurott-Hisui
Chevalier (Basculegion)
Maushold-Four
Gallade (M)
Mega Froslass
Bellibolt (

In [15]:
for k in POKEMON_DICT.keys():
    if "Floette" in k:
        print(k)

Floette
Floette-Mega


## 6. Model

Per-Pokemon vector =
`[species_emb | species_rdv | ability_emb | item_emb | item_rdv | moveset_emb | moveset_rdv]`.
The masked slot's vector is replaced by a learned mask token. Vectors are projected to `D_MODEL`,
passed through a Transformer encoder (order-free / no positional encoding, since a team is a set),
and the masked slot's output feeds four prediction heads.

In [16]:
class MaskedTeamTransformer(nn.Module):
    """Masked team model with optional legality-constrained, top-k/p decoding.

    `forward` returns raw logits (used for losses / backprop, unchanged).
    `predict` performs species selection (argmax or top-k/p sampling) and then
    constrains ability/moveset to what the chosen species can legally have
    (per POKEMON_DICT), self-correcting illegal predictions.
    """
    def __init__(self):
        super().__init__()
        self.species_emb = nn.Embedding(N_SPECIES, EMB["species"])
        self.ability_emb = nn.Embedding(N_ABILITY, EMB["ability"])
        self.item_emb    = nn.Embedding(N_ITEM,    EMB["item"])
        self.move_emb    = nn.Embedding(N_MOVE,    EMB["move"])

        self.use_move_attn = USE_MOVE_ATTENTION
        if self.use_move_attn:
            self.move_attn_emb = nn.MultiheadAttention(
                EMB["move"], MOVE_ATTENTION_HEADS, batch_first=True)
            self.move_attn_rdv = nn.MultiheadAttention(
                MOVE_RDV_DIM, MOVE_ATTENTION_HEADS, batch_first=True)

        self.input_dim = (EMB["species"] + SPECIES_RDV_DIM + EMB["ability"]
                          + EMB["item"] + ITEM_RDV_DIM
                          + EMB["move"] + MOVE_RDV_DIM)
        self.mask_token = nn.Parameter(torch.randn(self.input_dim) * 0.02)
        self.input_proj = nn.Linear(self.input_dim, D_MODEL)

        enc = nn.TransformerEncoderLayer(
            d_model=D_MODEL, nhead=N_HEADS, dim_feedforward=DIM_FEEDFORWARD,
            dropout=DROPOUT, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc, num_layers=N_LAYERS)

        self.head_species = nn.Linear(D_MODEL, N_SPECIES)
        self.head_ability = nn.Linear(D_MODEL, N_ABILITY)
        self.head_item    = nn.Linear(D_MODEL, N_ITEM)
        self.head_moves   = nn.Linear(D_MODEL, N_MOVE)   # multi-label moveset

        # ---- Legality matrices from POKEMON_DICT ----
        # ability_legal[s, a] / move_legal[s, m] = True if species s can have it.
        # Species absent from POKEMON_DICT (UNK / scraped variants) -> unconstrained.
        ability_legal = torch.zeros(N_SPECIES, N_ABILITY, dtype=torch.bool)
        move_legal    = torch.zeros(N_SPECIES, N_MOVE,    dtype=torch.bool)
        for name, sidx in SPECIES_VOCAB.items():
            entry = POKEMON_DICT.get(name)
            if entry is None:
                ability_legal[sidx] = True
                move_legal[sidx]    = True
                continue
            abils = entry.get("ability")
            abils = [abils] if isinstance(abils, str) else list(abils)
            a_idx = [ABILITY_VOCAB[a] for a in abils if a in ABILITY_VOCAB]
            if a_idx:
                ability_legal[sidx, a_idx] = True
            else:
                ability_legal[sidx] = True            # nothing mappable -> allow all
            m_idx = [MOVE_VOCAB[m] for m in entry.get("moves", []) if m in MOVE_VOCAB]
            if m_idx:
                move_legal[sidx, m_idx] = True
            else:
                move_legal[sidx] = True
        self.register_buffer("ability_legal", ability_legal)
        self.register_buffer("move_legal",    move_legal)

        # ---- Team-construction constraint tables ----
        # family_id groups a base species with its Mega form(s) so either form
        # blocks the other; is_mega flags Mega forms; form_to_stone_item maps a
        # Mega form -> the ITEM_VOCAB idx of the stone that enables it.
        mega_forms    = set(MEGA_STONE_TO_FORM.values())
        form_to_stone = {v: k for k, v in MEGA_STONE_TO_FORM.items()}
        is_mega   = torch.zeros(N_SPECIES, dtype=torch.bool)
        fts_item  = torch.full((N_SPECIES,), -1, dtype=torch.long)
        family_id = torch.zeros(N_SPECIES, dtype=torch.long)
        fam_lookup = {}
        for name, sidx in SPECIES_VOCAB.items():
            base = name.rsplit("-Mega", 1)[0] if name in mega_forms else name
            family_id[sidx] = fam_lookup.setdefault(base, len(fam_lookup))
            if name in mega_forms:
                is_mega[sidx] = True
                stone = form_to_stone.get(name)
                if stone in ITEM_VOCAB:
                    fts_item[sidx] = ITEM_VOCAB[stone]
        self.register_buffer("is_mega", is_mega)
        self.register_buffer("family_id", family_id)
        self.register_buffer("form_to_stone_item", fts_item)

        # ---- Mega Stone Assertion: each Mega Stone is legal ONLY for its base
        # species and its Mega form. Non-Mega-Stone items remain legal for all.
        item_legal = torch.ones(N_SPECIES, N_ITEM, dtype=torch.bool)
        for stone, form in MEGA_STONE_TO_FORM.items():
            if stone not in ITEM_VOCAB:
                continue
            iidx = ITEM_VOCAB[stone]
            base_name = form.rsplit("-Mega", 1)[0]
            legal_sp = set()
            if form in SPECIES_VOCAB:      legal_sp.add(SPECIES_VOCAB[form])
            if base_name in SPECIES_VOCAB: legal_sp.add(SPECIES_VOCAB[base_name])
            disallow = torch.ones(N_SPECIES, dtype=torch.bool)
            for s in legal_sp: disallow[s] = False
            item_legal[disallow, iidx] = False
        self.register_buffer("item_legal", item_legal)

    # ---------- shared encoding ----------
    def _moveset(self, move_idx, move_rdv):
        B = move_idx.size(0)
        me = self.move_emb(move_idx)                       # [B,6,4,move_dim]
        if self.use_move_attn:
            e = me.reshape(B * 6, 4, EMB["move"])
            e, _ = self.move_attn_emb(e, e, e)
            me = e.reshape(B, 6, 4, EMB["move"])
            r = move_rdv.reshape(B * 6, 4, MOVE_RDV_DIM)
            r, _ = self.move_attn_rdv(r, r, r)
            move_rdv = r.reshape(B, 6, 4, MOVE_RDV_DIM)
        return me.mean(dim=2), move_rdv.mean(dim=2)

    def forward(self, batch):
        sp = self.species_emb(batch["species_idx"])
        ab = self.ability_emb(batch["ability_idx"])
        it = self.item_emb(batch["item_idx"])
        ms_emb, ms_rdv = self._moveset(batch["move_idx"], batch["move_rdv"])
        x = torch.cat([sp, batch["species_rdv"], ab,
                       it, batch["item_rdv"], ms_emb, ms_rdv], dim=-1)
        B = x.size(0)
        mpos = batch["mask_pos"]
        x = x.clone()
        x[torch.arange(B), mpos] = self.mask_token
        h = self.encoder(self.input_proj(x))
        hm = h[torch.arange(B), mpos]
        return {
            "species": self.head_species(hm),
            "ability": self.head_ability(hm),
            "item":    self.head_item(hm),
            "moves":   self.head_moves(hm),
        }

    # ---------- decoding helpers ----------
    @staticmethod
    def _filter_logits(logits, top_k, top_p):
        """Return logits with non-top-k / non-nucleus entries set to -inf."""
        logits = logits.clone()
        if top_k and top_k >= 1:
            k = min(int(top_k), logits.size(-1))
            kth = logits.topk(k, dim=-1).values[..., -1, None]
            logits = logits.masked_fill(logits < kth, float("-inf"))
        if top_p and 0.0 < top_p < 1.0:
            s_logits, s_idx = torch.sort(logits, descending=True, dim=-1)
            cum = s_logits.softmax(-1).cumsum(-1)
            s_remove = cum > top_p
            s_remove[..., 1:] = s_remove[..., :-1].clone()  # keep the crossing token
            s_remove[..., 0] = False
            remove = torch.zeros_like(s_remove).scatter(-1, s_idx, s_remove)
            logits = logits.masked_fill(remove, float("-inf"))
        return logits

    @staticmethod
    def _sampling_active(phase):
        if not ((TOP_K and TOP_K >= 1) or (TOP_P and 0.0 < TOP_P < 1.0)):
            return False
        return SAMPLING_PHASE == "both" or SAMPLING_PHASE == phase

    @torch.no_grad()
    def predict(self, batch, phase="predict", n_predictions=1, n_moves=4):
        """Decode predictions for the masked slot.

        Returns top-`n_predictions` candidate species (constrained by team rules
        and team-mate uniqueness), each accompanied by an ability / item / moveset
        derived from that species' legal pool.
            * species [B, n]   * ability [B, n]   * item [B, n]
            * moves   list[B] of list[n] of tensor[k]   * logits raw model output
        n=1 reproduces the original single-prediction behaviour (the only path
        that respects top-k/p sampling); n>1 always uses deterministic top-k from
        the constrained logits, which gives stable diverse candidates.

        Enforces:
            (1) no Mega prediction if >=2 visible Pokemon are already Megas
            (2) no duplicate / base<->Mega of a visible Pokemon
            (3) a predicted Mega forces its enabling Mega Stone as the item
            (4) Mega Stone items are only legal for their owning species
            (5) optional ENFORCE_LEGALITY restricts ability/moves to those the
                predicted species can actually learn (POKEMON_DICT).
        """
        out = self.forward(batch)
        B, S = out["species"].shape
        active = self._sampling_active(phase) and n_predictions == 1
        sp_logits = out["species"].clone()

        # ----- team-construction constraints on the species choice -----
        pos  = torch.arange(6, device=sp_logits.device)
        keep = pos.unsqueeze(0) != batch["mask_pos"].unsqueeze(1)        # [B,6]
        vis  = batch["species_idx"][keep].view(B, 5)
        vis_fam = self.family_id[vis]                                     # [B,5]
        forbid  = (self.family_id.view(1, S, 1) == vis_fam.view(B, 1, 5)).any(-1)
        two_megas = self.is_mega[vis].sum(1) >= 2
        forbid = forbid | (two_megas.view(B, 1) & self.is_mega.view(1, S))
        sp_logits = sp_logits.masked_fill(forbid, float("-inf"))
        dead = torch.isinf(sp_logits).all(-1)
        if dead.any():
            sp_logits[dead] = out["species"][dead]

        if active:
            probs = F.softmax(self._filter_logits(sp_logits, TOP_K, TOP_P) / TEMP, -1)
            sp_cands = torch.multinomial(probs, 1)                       # [B,1]
        else:
            k = min(int(n_predictions), S)
            sp_cands = sp_logits.topk(k, dim=-1).indices                 # [B,n]
        n = sp_cands.size(1)

        ab_cands = torch.zeros_like(sp_cands)
        it_cands = torch.zeros_like(sp_cands)
        mv_cands = [[None] * n for _ in range(B)]

        for j in range(n):
            sp_j = sp_cands[:, j]
            ab_logits = out["ability"].clone()
            mv_logits = out["moves"].clone()
            it_logits = out["item"].clone()
            if ENFORCE_LEGALITY:
                ab_logits = ab_logits.masked_fill(~self.ability_legal[sp_j], float("-inf"))
                mv_logits = mv_logits.masked_fill(~self.move_legal[sp_j],    float("-inf"))
            # Mega Stone Assertion: applies always
            it_logits = it_logits.masked_fill(~self.item_legal[sp_j], float("-inf"))

            if active:
                ab_cands[:, j] = torch.multinomial(ab_logits.softmax(-1), 1).squeeze(-1)
            else:
                ab_cands[:, j] = ab_logits.argmax(-1)

            it_j = it_logits.argmax(-1)
            forced = self.form_to_stone_item[sp_j]                       # -1 if N/A
            it_j = torch.where(forced >= 0, forced, it_j)
            it_cands[:, j] = it_j

            for i in range(B):
                legal = torch.isfinite(mv_logits[i])
                kk = int(min(n_moves, int(legal.sum().item()))) or 1
                if active:
                    mv_cands[i][j] = torch.multinomial(
                        mv_logits[i].softmax(-1), kk, replacement=False)
                else:
                    mv_cands[i][j] = mv_logits[i].topk(kk).indices

        return {"species": sp_cands, "ability": ab_cands, "item": it_cands,
                "moves": mv_cands, "logits": out}

    # ----- Hidden-state / embedding extraction -----
    @torch.no_grad()
    def get_embedding(self, name, kind):
        """Return the learned embedding vector for a token by name.
        kind: 'species' | 'ability' | 'item' | 'move'."""
        table = {
            "species": (SPECIES_VOCAB, self.species_emb),
            "ability": (ABILITY_VOCAB, self.ability_emb),
            "item":    (ITEM_VOCAB,    self.item_emb),
            "move":    (MOVE_VOCAB,    self.move_emb),
        }
        if kind not in table:
            raise ValueError(f"kind must be one of {list(table)}, got {kind!r}")
        vocab, emb = table[kind]
        if name not in vocab:
            raise KeyError(f"{name!r} not found in {kind} vocab")
        idx = torch.tensor(vocab[name], device=emb.weight.device)
        return emb(idx).detach().clone()

    def training_masked_logits(self, out, batch):
        """Ability/move logits restricted to the TRUE species' legal set, with
        the ground-truth index always kept legal (keeps CE/BCE finite). Used so
        the heads learn within the legal subspace when SAMPLING_PHASE covers
        training."""
        sp_true = batch["y_species"]
        a_mask = self.ability_legal[sp_true].clone()
        a_mask[torch.arange(a_mask.size(0)), batch["y_ability"]] = True
        m_mask = self.move_legal[sp_true].clone()
        m_mask = m_mask | (batch["y_moves"] > 0)
        # CE is -inf-safe (true class kept legal); BCE is NOT: a -inf logit with
        # a 0 target evaluates 0*inf -> nan, so moves use a large finite negative.
        ab = out["ability"].masked_fill(~a_mask, float("-inf"))
        mv = out["moves"].masked_fill(~m_mask, -1e9)
        return ab, mv

## 7. Training & Validation Pipeline

In [17]:
ce  = nn.CrossEntropyLoss()
bce = nn.BCEWithLogitsLoss()

def best_of_n_ce(logits, target, n):
    """For n=1 this is plain CE (identical to prior behavior).
    For n>1 the loss is zeroed on samples where `target` is already among the
    model's top-n predicted classes -- i.e. the model is rewarded for getting
    the answer within its top-n shortlist, and only updates weights on
    samples it got wrong on all n attempts."""
    if n <= 1:
        return F.cross_entropy(logits, target)
    topn = logits.topk(min(n, logits.size(-1)), dim=-1).indices
    correct = (topn == target.unsqueeze(-1)).any(-1)
    ce_per = F.cross_entropy(logits, target, reduction="none")
    return (ce_per * (~correct).float()).mean()

def compute_loss(model, out, batch, phase):
    """Species/item: best-of-n CE on full logits. Ability/moves: if the sampling
    pipeline is active for `phase` (or train-time legality is enabled), the
    legality-masked logits are used so the heads learn within the legal subspace."""
    n = TRAIN_PREDICTIONS if phase == "train" else TEST_PREDICTIONS
    if model._sampling_active(phase) or (ENFORCE_LEGALITY and phase == "train"
                                         and SAMPLING_PHASE in ("train", "both")):
        ab_logits, mv_logits = model.training_masked_logits(out, batch)
    else:
        ab_logits, mv_logits = out["ability"], out["moves"]
    sp_loss = best_of_n_ce(out["species"], batch["y_species"], n)
    it_loss = best_of_n_ce(out["item"],    batch["y_item"],    n)
    ab_loss = best_of_n_ce(ab_logits,      batch["y_ability"], n)
    mv_loss = bce(mv_logits, batch["y_moves"])
    return sp_loss + ab_loss + it_loss + mv_loss

def to_device(batch, device):
    return {k: v.to(device) for k, v in batch.items()}

@torch.no_grad()
def evaluate(model, loader, device, phase="predict"):
    """Loss on raw logits (best-of-TEST_PREDICTIONS). Accuracy via the
    constrained `predict` decode: each head is "correct" if the true label is
    among the n candidate predictions for that head."""
    n = TEST_PREDICTIONS
    model.eval()
    cnt = 0
    correct = {"species": 0, "ability": 0, "item": 0}
    move_recall = 0.0
    total_loss = 0.0
    for batch in loader:
        batch = to_device(batch, device)
        out = model.forward(batch)
        total_loss += compute_loss(model, out, batch, phase).item() * batch["y_species"].size(0)
        pred = model.predict(batch, phase=phase, n_predictions=n)
        for key in correct:
            hit = (pred[key] == batch[f"y_{key}"].unsqueeze(-1)).any(-1)
            correct[key] += hit.sum().item()
        for i in range(batch["y_moves"].size(0)):
            true_idx = set(batch["y_moves"][i].nonzero(as_tuple=True)[0].tolist())
            best = 0.0
            for mv in pred["moves"][i]:
                hit = len(set(mv.tolist()) & true_idx)
                best = max(best, hit / max(len(true_idx), 1))
            move_recall += best
        cnt += batch["y_species"].size(0)
    return {
        "loss":        total_loss / max(cnt, 1),
        "species_acc": correct["species"] / max(cnt, 1),
        "ability_acc": correct["ability"] / max(cnt, 1),
        "item_acc":    correct["item"]    / max(cnt, 1),
        "move_recall": move_recall / max(cnt, 1),
    }

def train(model, train_ds, test_ds, epochs=EPOCHS, batch_size=BATCH_SIZE,
          lr=LR, device=DEVICE):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    tl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    vl = DataLoader(test_ds,  batch_size=batch_size, shuffle=False)
    for ep in range(1, epochs + 1):
        model.train()
        run = 0.0
        for batch in tl:
            batch = to_device(batch, device)
            opt.zero_grad()
            loss = compute_loss(model, model.forward(batch), batch, "train")
            loss.backward()
            opt.step()
            run += loss.item() * batch["y_species"].size(0)
        val = evaluate(model, vl, device, phase="predict")
        print(f"epoch {ep:2d} | train_loss {run/len(train_ds):.4f} "
              f"| val_loss {val['loss']:.4f} "
              f"| spc {val['species_acc']:.2f} ab {val['ability_acc']:.2f} "
              f"it {val['item_acc']:.2f} mv {val['move_recall']:.2f}")
    return model

## 7.5 Gridsearch Setup

Two independent gridsearches with their own toggles. Both default to OFF, in which case
Section 8 trains the model exactly as it did before. Toggle either (or both) on and
Section 8 will instead iterate the corresponding grid and produce a results table.

In [18]:
import itertools
import pandas as pd

# ---- Toggles (read by Section 8) -----------------------------------------
RUN_ARCH_GRIDSEARCH     = False
RUN_BEHAVIOR_GRIDSEARCH = False

# ---- Grid 1: Transformer architecture ------------------------------------
# Every other knob held to its plain-vanilla default (predict 1 Pokemon,
# no top-k/p sampling, T=1) so only architecture is varied.
ARCH_GRID = {
    "D_MODEL":         [128, 256, 512],
    "N_HEADS":         [4, 6, 8],
    "N_LAYERS":        [2, 3, 4],
    "DIM_FEEDFORWARD": [256, 512],
}
ARCH_BASE_OVERRIDES = {
    "TRAIN_PREDICTIONS": 1, "TEST_PREDICTIONS": 1,
    "TOP_K": 0, "TOP_P": 0, "TEMP": 1.0,
}

# ---- Grid 2: Model behavior ----------------------------------------------
# Architecture held to whatever the notebook is currently configured for.
# TOP_K and TOP_P share the same sweep list `TOPKP_VALUES`: a value >=1 is
# routed to TOP_K (TOP_P=0), a value in (0,1) is routed to TOP_P (TOP_K=0).
TOPKP_VALUES = [6, 8, 10, 0.2, 0.4, 0.6, 0.8]
TOPP_VALUES = [0.2, 0.4, 0.6, 0.8]
BEHAVIOR_GRID = {
    "TRAIN_PREDICTIONS": [2, 3, 4],
    "TEST_PREDICTIONS":  [2,3,4],
    "TOPKP":             TOPP_VALUES,
    "TEMP":              [1.0],
    "SAMPLING_PHASE" : ["train"]
}
FULL_BGRID = {
    "TRAIN_PREDICTIONS": [1, 2, 3, 4],
    "TEST_PREDICTIONS":  [1, 2, 3, 4],
    "TOPKP":             TOPKP_VALUES,
    "TEMP":              [1.0, 2.0, 4.0, 8.0],
    "SAMPLING_PHASE" : ["train", "predict", "both"]
}

def _kp_to_overrides(v):
    """Route a TOPKP grid value into TOP_K/TOP_P overrides."""
    return ({"TOP_K": int(v), "TOP_P": 0} if v >= 1
            else {"TOP_K": 0, "TOP_P": float(v)})

def _expand_arch_grid():
    return [{"D_MODEL": d, "N_HEADS": h, "N_LAYERS": L, "DIM_FEEDFORWARD": ff}
            for d, h, L, ff in itertools.product(
                ARCH_GRID["D_MODEL"], ARCH_GRID["N_HEADS"],
                ARCH_GRID["N_LAYERS"], ARCH_GRID["DIM_FEEDFORWARD"])]

def _expand_behavior_grid():
    rows = []
    for tp, tep, kp, t, sf in itertools.product(
            BEHAVIOR_GRID["TRAIN_PREDICTIONS"], BEHAVIOR_GRID["TEST_PREDICTIONS"],
            BEHAVIOR_GRID["TOPKP"],            BEHAVIOR_GRID["TEMP"],
            BEHAVIOR_GRID["SAMPLING_PHASE"]):
        ov = _kp_to_overrides(kp)
        rows.append({"TRAIN_PREDICTIONS": tp, "TEST_PREDICTIONS": tep,
                     "TOP_K": ov["TOP_K"], "TOP_P": ov["TOP_P"], "TEMP": t,
                     "SAMPLING_PHASE" : sf})
    return rows

def _train_one_silent(epochs, batch_size, lr, device, train_ds, test_ds):
    """Train a freshly-instantiated model on the current globals and return
    (final_train_loss, val_metrics)."""
    set_seed(SEED)
    m = MaskedTeamTransformer().to(device)
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    tl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    vl = DataLoader(test_ds,  batch_size=batch_size, shuffle=False)
    final = 0.0
    for _ep in range(epochs):
        m.train()
        running = 0.0
        for batch in tl:
            batch = to_device(batch, device)
            opt.zero_grad()
            loss = compute_loss(m, m.forward(batch), batch, "train")
            loss.backward()
            opt.step()
            running += loss.item() * batch["y_species"].size(0)
        final = running / max(len(train_ds), 1)
    val = evaluate(m, vl, device, phase="predict")
    return final, val

def run_gridsearch(rows, train_ds, test_ds,
                   epochs=None, batch_size=None, lr=None, device=None,
                   extra_overrides=None, log_name="gridsearch"):
    """Iterate `rows` (each a dict of globals to override), train each config,
    and return a pandas DataFrame with the settings + metrics. Invalid combos
    (e.g. D_MODEL % N_HEADS != 0) are recorded with status='skipped'."""
    g = globals()
    epochs     = g["EPOCHS"]     if epochs     is None else epochs
    batch_size = g["BATCH_SIZE"] if batch_size is None else batch_size
    lr         = g["LR"]         if lr         is None else lr
    device     = g["DEVICE"]     if device     is None else device

    results = []
    extra_overrides = extra_overrides or {}
    print(f"[{log_name}] {len(rows)} configs x {epochs} epochs on {device}")
    for i, row in enumerate(rows, 1):
        all_ov = {**extra_overrides, **row}

        # Hard-skip transformer-impossible combos
        d = all_ov.get("D_MODEL", g["D_MODEL"])
        h = all_ov.get("N_HEADS", g["N_HEADS"])
        if d % h != 0:
            results.append({**row, "train_loss": float("nan"),
                            "val_loss":    float("nan"),
                            "species_acc": float("nan"),
                            "ability_acc": float("nan"),
                            "item_acc":    float("nan"),
                            "move_recall": float("nan"),
                            "status": "skipped: D_MODEL % N_HEADS != 0"})
            print(f"  [{i:>3}/{len(rows)}] SKIP {row}")
            continue

        saved = {k: g.get(k) for k in all_ov}
        try:
            g.update(all_ov)
            tl_loss, vmet = _train_one_silent(
                epochs, batch_size, lr, device, train_ds, test_ds)
            results.append({**row,
                "train_loss":  round(tl_loss, 4),
                "val_loss":    round(vmet["loss"], 4),
                "species_acc": round(vmet["species_acc"], 4),
                "ability_acc": round(vmet["ability_acc"], 4),
                "item_acc":    round(vmet["item_acc"], 4),
                "move_recall": round(vmet["move_recall"], 4),
                "status":      "ok"})
            print(f"  [{i:>3}/{len(rows)}] {row} -> "
                  f"tl={tl_loss:.3f} vl={vmet['loss']:.3f} "
                  f"spc={vmet['species_acc']:.2f} ab={vmet['ability_acc']:.2f} "
                  f"it={vmet['item_acc']:.2f} mv={vmet['move_recall']:.2f}")
        except Exception as e:
            results.append({**row, "train_loss": float("nan"),
                            "val_loss":    float("nan"),
                            "species_acc": float("nan"),
                            "ability_acc": float("nan"),
                            "item_acc":    float("nan"),
                            "move_recall": float("nan"),
                            "status": f"error: {type(e).__name__}: {e}"})
            print(f"  [{i:>3}/{len(rows)}] ERROR {row}: {e}")
        finally:
            g.update(saved)
    return pd.DataFrame(results)

## 8. Run

If `RUN_ARCH_GRIDSEARCH` and `RUN_BEHAVIOR_GRIDSEARCH` are both `False`, this trains
the model normally (debug or full, controlled by `DEBUG_RUN`). If either toggle is
on, the corresponding gridsearch runs instead, and the per-configuration results
table is printed and saved to CSV.

In [23]:
set_seed(SEED)

train_ds = MaskedTeamDataset(train_teams, TRAIN_MASK_STRATEGY)
test_ds  = MaskedTeamDataset(test_teams,  EVAL_MASK_STRATEGY)
print(f"datasets -> train {len(train_ds)}, test {len(test_ds)}")

if RUN_ARCH_GRIDSEARCH or RUN_BEHAVIOR_GRIDSEARCH:
    if RUN_ARCH_GRIDSEARCH:
        df_arch = run_gridsearch(
            _expand_arch_grid(), train_ds, test_ds,
            extra_overrides=ARCH_BASE_OVERRIDES,
            log_name="ARCH GRIDSEARCH")
        print("\nARCH GRIDSEARCH RESULTS\n" + "=" * 80)
        print(df_arch.to_string(index=False))
        os.makedirs(RESULTS_DIR, exist_ok=True)
        df_arch.to_csv(os.path.join(RESULTS_DIR, "gridsearch_arch_results.csv"), index=False)
        print(f"saved -> {os.path.join(RESULTS_DIR, 'gridsearch_arch_results.csv')}")

    if RUN_BEHAVIOR_GRIDSEARCH:
        df_beh = run_gridsearch(
            _expand_behavior_grid(), train_ds, test_ds,
            log_name="BEHAVIOR GRIDSEARCH")
        print("\nBEHAVIOR GRIDSEARCH RESULTS\n" + "=" * 80)
        print(df_beh.to_string(index=False))
        os.makedirs(RESULTS_DIR, exist_ok=True)
        df_beh.to_csv(os.path.join(RESULTS_DIR, "gridsearch_behavior_results_tesamp.csv"), index=False)
        print(f"saved -> {os.path.join(RESULTS_DIR, 'gridsearch_behavior_results_tesamp.csv')}")
else:
    # ---- Standard training (both toggles off) ----
    model = MaskedTeamTransformer()
    print("per-Pokemon input dim:", model.input_dim)
    _sample = next(iter(DataLoader(train_ds, batch_size=4)))
    _out = model(to_device(model.cpu() and _sample, "cpu"))
    print("output shapes:", {k: tuple(v.shape) for k, v in _out.items()})

    model = train(model, train_ds, test_ds, epochs=EPOCHS)
    print("\nFinal:", evaluate(model, DataLoader(test_ds, batch_size=BATCH_SIZE), DEVICE))

datasets -> train 8267, test 1222
per-Pokemon input dim: 188
output shapes: {'species': (4, 336), 'ability': (4, 269), 'item': (4, 209), 'moves': (4, 683)}
epoch  1 | train_loss 7.5317 | val_loss 8.4648 | spc 0.55 ab 0.54 it 0.35 mv 0.51
epoch  2 | train_loss 6.3991 | val_loss 7.9221 | spc 0.61 ab 0.56 it 0.43 mv 0.55
epoch  3 | train_loss 5.8778 | val_loss 8.1199 | spc 0.62 ab 0.57 it 0.43 mv 0.56
epoch  4 | train_loss 5.6297 | val_loss 7.9266 | spc 0.65 ab 0.59 it 0.46 mv 0.58
epoch  5 | train_loss 5.4166 | val_loss 8.1664 | spc 0.65 ab 0.59 it 0.45 mv 0.58
epoch  6 | train_loss 5.3373 | val_loss 8.0697 | spc 0.64 ab 0.58 it 0.44 mv 0.59
epoch  7 | train_loss 5.1057 | val_loss 7.8654 | spc 0.64 ab 0.58 it 0.45 mv 0.58
epoch  8 | train_loss 5.1401 | val_loss 7.9177 | spc 0.66 ab 0.60 it 0.48 mv 0.59
epoch  9 | train_loss 5.0498 | val_loss 8.2520 | spc 0.65 ab 0.58 it 0.47 mv 0.59
epoch 10 | train_loss 4.8933 | val_loss 8.1139 | spc 0.66 ab 0.59 it 0.45 mv 0.59
epoch 11 | train_loss 4.

## 9. Saving / Loading the Model

`save_checkpoint` bundles the trained weights together with the architecture config and the
vocabularies, so the model can be rebuilt for inference later without re-deriving anything.
`load_checkpoint` reconstructs it. The debug-trained model is saved below.

In [25]:
MODEL_NAME     = "masked_team_transformer.pt"
FULL_MODEL_DIR = os.path.join(MODEL_DIR, MODEL_NAME)

def save_checkpoint(model, path=FULL_MODEL_DIR):
    torch.save({
        "state_dict": model.state_dict(),
        "config": {
            "EMB": EMB,
            "SPECIES_RDV_DIM": SPECIES_RDV_DIM,
            "ITEM_RDV_DIM": ITEM_RDV_DIM,
            "MOVE_RDV_DIM": MOVE_RDV_DIM,
            "D_MODEL": D_MODEL, "N_HEADS": N_HEADS, "N_LAYERS": N_LAYERS,
            "DIM_FEEDFORWARD": DIM_FEEDFORWARD, "DROPOUT": DROPOUT,
            "USE_MOVE_ATTENTION": USE_MOVE_ATTENTION,
            "MOVE_ATTENTION_HEADS": MOVE_ATTENTION_HEADS,
            "embed_config_name": EMBED_CONFIG_NAME,
        },
        "vocabs": {
            "species": SPECIES_VOCAB, "ability": ABILITY_VOCAB,
            "item": ITEM_VOCAB, "move": MOVE_VOCAB,
        },
    }, path)
    print(f"saved checkpoint -> {path} ({os.path.getsize(path)/1e6:.2f} MB)")

def load_checkpoint(path=FULL_MODEL_DIR, device=DEVICE):
    """Rebuilds the model from a checkpoint. Assumes the architecture/vocab
    globals in this notebook match those stored in the checkpoint."""
    ckpt = torch.load(path, map_location=device, weights_only=False)
    model = MaskedTeamTransformer()
    model.load_state_dict(ckpt["state_dict"])
    model.to(device).eval()
    print(f"loaded checkpoint <- {path} "
          f"(trained config: {ckpt['config']['embed_config_name']})")
    return model, ckpt

if "model" in globals():
    # Save the trained model, then verify it reloads and matches
    save_checkpoint(model)
    _reloaded, _ckpt = load_checkpoint()
    _chk = evaluate(_reloaded, DataLoader(test_ds, batch_size=BATCH_SIZE), DEVICE)
    print("reloaded model eval:", _chk)
else:
    print("skip: no single `model` to save (Section 8 ran a gridsearch).")

saved checkpoint -> /content/drive/Shared drives/ML2 Final Project/Models/masked_team_transformer.pt (3.66 MB)
loaded checkpoint <- /content/drive/Shared drives/ML2 Final Project/Models/masked_team_transformer.pt (trained config: medium)
reloaded model eval: {'loss': 8.959024549506104, 'species_acc': 0.6653027823240589, 'ability_acc': 0.6047463175122749, 'item_acc': 0.4762684124386252, 'move_recall': 0.6088379705400983}


## 10. Inspect Sample Predictions

Set `N_SAMPLES` to how many random validation teams you want to inspect. For each, the masked slot's
predicted species / ability / item / moveset is shown next to the ground truth. Moves are multi-label,
so the top-k predictions (k = number of true moves) are shown.

In [26]:
N_SAMPLES = 5   # how many random validation teams to inspect

INV_SPECIES = {v: k for k, v in SPECIES_VOCAB.items()}
INV_ABILITY = {v: k for k, v in ABILITY_VOCAB.items()}
INV_ITEM    = {v: k for k, v in ITEM_VOCAB.items()}
INV_MOVE    = {v: k for k, v in MOVE_VOCAB.items()}

@torch.no_grad()
def show_predictions(model, dataset, n=N_SAMPLES, n_predictions=None, device=DEVICE):
    """Uses model.predict with `n_predictions` candidates (defaults to TEST_PREDICTIONS).
    A head is marked OK if the true label is among the listed candidates."""
    if n_predictions is None:
        n_predictions = TEST_PREDICTIONS
    model.eval()
    n = min(n, len(dataset))
    idxs = random.sample(range(len(dataset)), n)
    for s, di in enumerate(idxs, 1):
        sample = dataset[di]
        batch = {k: v.unsqueeze(0).to(device) for k, v in sample.items()}
        pred = model.predict(batch, phase="predict", n_predictions=n_predictions)
        mpos = sample["mask_pos"].item()

        sp_cs = [INV_SPECIES[i] for i in pred["species"][0].tolist()]
        ab_cs = [INV_ABILITY[i] for i in pred["ability"][0].tolist()]
        it_cs = [INV_ITEM[i]    for i in pred["item"][0].tolist()]
        mv_cs = [[INV_MOVE[i] for i in m.tolist()] for m in pred["moves"][0]]

        t_sp = INV_SPECIES[sample["y_species"].item()]
        t_ab = INV_ABILITY[sample["y_ability"].item()]
        t_it = INV_ITEM[sample["y_item"].item()]
        t_mv = [INV_MOVE[i] for i in sample["y_moves"].nonzero(as_tuple=True)[0].tolist()]

        team    = dataset.teams[di]
        visible = [p[0] for j, p in enumerate(team) if j != mpos]
        mark = lambda ok: "OK " if ok else "X  "
        print(f"=== Sample {s}  (test team #{di}, masked slot {mpos}, n_preds={n_predictions}) ===")
        print("  given 5  :", ", ".join(visible))
        if n_predictions == 1:
            print(f"  species  : {mark(t_sp in sp_cs)}pred={sp_cs[0]:<22} true={t_sp}")
            print(f"  ability  : {mark(t_ab in ab_cs)}pred={ab_cs[0]:<22} true={t_ab}")
            print(f"  item     : {mark(t_it in it_cs)}pred={it_cs[0]:<22} true={t_it}")
            print(f"  moves    : pred={mv_cs[0]}")
        else:
            print(f"  species  : {mark(t_sp in sp_cs)}preds={sp_cs}   true={t_sp}")
            print(f"  ability  : {mark(t_ab in ab_cs)}preds={ab_cs}   true={t_ab}")
            print(f"  item     : {mark(t_it in it_cs)}preds={it_cs}   true={t_it}")
            for k, mv in enumerate(mv_cs, 1):
                print(f"  moves #{k}: pred={mv}")
        print(f"             true={t_mv}")
        print()

if "model" in globals():
    show_predictions(model, test_ds, N_SAMPLES)
else:
    print("skip: no `model` to inspect (Section 8 ran a gridsearch).")

=== Sample 1  (test team #682, masked slot 5, n_preds=6) ===
  given 5  : Sneasler, Kingambit, Whimsicott, Charizard-Mega-Y, Wash Rotom
  species  : OK preds=['Garchomp', 'Basculegion', 'Dragapult', 'Basculegion (M)', 'Aerodactyl-Mega', 'Mamoswine']   true=Garchomp
  ability  : OK preds=['Rough Skin', 'Adaptability', 'Clear Body', 'Adaptability', 'Tough Claws', 'Oblivious']   true=Rough Skin
  item     : X  preds=['Choice Scarf', 'Choice Scarf', 'Choice Scarf', 'Choice Scarf', 'Aerodactylite', 'Choice Scarf']   true=Soft Sand
  moves #1: pred=['Dragon Claw', 'Earthquake', 'Rock Slide', 'Stomping Tantrum']
  moves #2: pred=['Last Respects', 'Wave Crash', 'Flip Turn', 'Aqua Jet']
  moves #3: pred=['Dragon Claw', 'Protect', 'Shadow Ball', 'Dragon Darts']
  moves #4: pred=['Dragon Claw', 'Last Respects', 'Earthquake', 'Rock Slide']
  moves #5: pred=['Dragon Claw', 'Earthquake', 'Rock Slide', 'Protect']
  moves #6: pred=['Earthquake', 'Rock Slide', 'Stomping Tantrum', 'Protect']
           